# 01 — MacroDash NBB : Exploration des séries Eurostat

**Auteur :** Tahar Guenfoud | Mai 2026  
**Source :** Eurostat API JSON (`ec.europa.eu/eurostat`) — stat.nbb.be hors service

## Objectifs Jour 1

1. Vérifier que `EurostatClient` fonctionne sur les 5 séries cibles
2. Construire le DataFrame mensuel consolidé
3. Produire les premières visualisations **Plotly** interactives
4. Identifier la narrative bancaire : cycle BCE 2022-2024

## Séries cibles

| Clé interne | Code Eurostat | Description | Pertinence bancaire |
|-------------|--------------|-------------|--------------------|
| `olo_10y` | `irt_lt_mcby_m` | Taux OLO belge 10 ans | Coût financement long terme |
| `euribor_3m` | `irt_h_mr3_m` | EURIBOR 3M (zone euro) | Taux court terme BCE |
| `inflation` | `prc_hicp_manr` | Inflation IPCH annuelle | Pression remboursement |
| `chomage` | `une_rt_m` | Taux chômage mensuel | Solvabilité emprunteurs |
| `pib` | `nama_10_gdp` | PIB Belgique | Santé macro |


---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ajouter le dossier racine du projet au path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.config import DATASETS, BCE_EVENTS, DATA_RAW, DATA_PROCESSED
from src.fetch_eurostat import EurostatClient
from src.transform import build_macro_df

print(f"ROOT        : {ROOT}")
print(f"DATA_RAW    : {DATA_RAW}")
print(f"Datasets    : {list(DATASETS.keys())}")
print("Imports OK")

---
## 2. Récupération des données via Eurostat

La première exécution appelle l'API (~10 s). Les suivantes chargent les CSVs en cache.

In [ ]:
client = EurostatClient()

# Charger depuis le cache si disponible
cached = EurostatClient.load_cached()
if len(cached) == len(DATASETS):
    print(f"Cache complet ({len(cached)} séries) — chargement depuis data/raw/")
    series = cached
else:
    print(f"Cache incomplet ({len(cached)}/{len(DATASETS)}) — appel API Eurostat")
    series = client.fetch_all(start="2010", cache=True)

print()
print("INVENTAIRE")
print("-" * 45)
for k, df in series.items():
    cfg = DATASETS[k]
    if not df.empty:
        print(f"  {k:<15} {len(df):>4} obs  |  {df['periode'].min()} → {df['periode'].max()}")
    else:
        print(f"  {k:<15} VIDE")

---
## 3. Construction du DataFrame mensuel consolidé

In [ ]:
macro = build_macro_df(series)

print(f"Shape   : {macro.shape}")
print(f"Période : {macro.index.min().date()} → {macro.index.max().date()}")
print(f"NaN/col :\n{macro.isna().sum()}")
macro.tail(6)

---
## 4. Visualisation Plotly — Séries individuelles

Graphiques interactifs (zoom, hover, export PNG).

In [ ]:
def add_bce_annotations(fig, row=None, col=None):
    """Ajoute les lignes verticales des événements BCE sur une figure Plotly."""
    kw = {"row": row, "col": col} if row else {}
    for date_str, label, color in BCE_EVENTS:
        fig.add_vline(
            x=date_str,
            line_dash="dot",
            line_color=color,
            opacity=0.6,
            annotation_text=label,
            annotation_position="top right",
            annotation_font_size=9,
            annotation_font_color=color,
            **kw,
        )


# Subplot 4 séries en colonne
SERIES_TO_PLOT = [k for k in ("olo_10y", "euribor_3m", "inflation", "chomage") if k in macro.columns]

fig = make_subplots(
    rows=len(SERIES_TO_PLOT),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[DATASETS[k]["label"] for k in SERIES_TO_PLOT],
    vertical_spacing=0.06,
)

for i, key in enumerate(SERIES_TO_PLOT, start=1):
    cfg = DATASETS[key]
    s = macro[key].dropna()
    fig.add_trace(
        go.Scatter(
            x=s.index,
            y=s.values,
            mode="lines",
            name=cfg["label"],
            line=dict(color=cfg["color"], width=2),
            hovertemplate="%{x|%Y-%m}: %{y:.2f} " + cfg["unit"] + "<extra></extra>",
        ),
        row=i,
        col=1,
    )

fig.update_layout(
    height=280 * len(SERIES_TO_PLOT),
    title_text="Indicateurs macroéconomiques belges — 2010-2025 (Eurostat)",
    title_font_size=14,
    showlegend=False,
    template="plotly_white",
    margin=dict(t=80, b=40),
)

# Zone COVID
fig.add_vrect(
    x0="2020-03", x1="2021-06",
    fillcolor="gray", opacity=0.07,
    layer="below", line_width=0,
    annotation_text="COVID",
    annotation_position="top left",
    annotation_font_size=9,
)

fig.show()

---
## 5. Visualisation narrative — Cycle BCE 2022-2024

> *La BCE a relevé ses taux de 0 % à 4,5 % entre juillet 2022 et septembre 2023.*  
> *Ce graphique matérialise la transmission : OLO + EURIBOR montent ensemble,*  
> *l'inflation culmine puis redescend, le chômage reste stable.*

In [ ]:
df_plot = macro["2019":"2025"].copy()

fig2 = go.Figure()

COLOR_MAP = {
    "olo_10y":    ("OLO 10Y (%)",          "#1a5276", "solid"),
    "euribor_3m": ("EURIBOR 3M (%)",       "#c0392b", "dash"),
    "inflation":  ("Inflation IPCH (%)",   "#d35400", "dot"),
    "chomage":    ("Chômage (% actifs)",    "#6c3483", "dashdot"),
}

for key, (label, color, dash) in COLOR_MAP.items():
    if key not in df_plot.columns:
        continue
    s = df_plot[key].dropna()
    fig2.add_trace(go.Scatter(
        x=s.index, y=s.values,
        mode="lines",
        name=label,
        line=dict(color=color, width=2.2, dash=dash),
        hovertemplate="%{x|%Y-%m}: %{y:.2f}<extra></extra>",
    ))

# Annotations événements BCE
for date_str, label, color in BCE_EVENTS:
    fig2.add_vline(
        x=date_str, line_dash="dot", line_color=color, opacity=0.5,
        annotation_text=label, annotation_position="top right",
        annotation_font_size=9, annotation_font_color=color,
    )

fig2.add_vrect(
    x0="2020-03", x1="2021-06",
    fillcolor="gray", opacity=0.07, layer="below", line_width=0,
    annotation_text="COVID", annotation_position="top left",
    annotation_font_size=9,
)

fig2.update_layout(
    title="Cycle BCE 2022-2024 — Impact sur les indicateurs belges",
    xaxis_title="Période",
    yaxis_title="Valeur (%)",
    template="plotly_white",
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
)

fig2.show()

# Export PNG pour le README
try:
    out = DATA_PROCESSED / "narrative_bce_plotly.png"
    fig2.write_image(str(out), width=1200, height=480, scale=1.5)
    print(f"PNG sauvegardé : {out}")
except Exception as e:
    print(f"Export PNG ignoré (kaleido non installé) : {e}")

---
## 6. Matrice de corrélation (Plotly Heatmap)

In [ ]:
labels_map = {k: v["label"] for k, v in DATASETS.items()}
cols_available = [c for c in DATASETS if c in macro.columns]

corr = macro[cols_available].corr(method="pearson").round(3)
labels_display = [labels_map.get(c, c) for c in corr.columns]

fig3 = go.Figure(go.Heatmap(
    z=corr.values,
    x=labels_display,
    y=labels_display,
    colorscale="RdYlGn",
    zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate="%{text}",
    textfont_size=12,
    hovertemplate="%{y} × %{x} : %{z:.3f}<extra></extra>",
))

fig3.update_layout(
    title="Matrice de corrélation — Indicateurs macro belges (Pearson)",
    template="plotly_white",
    height=420,
    width=560,
    xaxis=dict(tickangle=-30),
)

fig3.show()

print("\nCorrélations fortes (|r| > 0.6) :")
for i in range(len(corr)):
    for j in range(i + 1, len(corr)):
        r = corr.values[i, j]
        if abs(r) > 0.6:
            print(f"  {corr.index[i]:<12} × {corr.columns[j]:<12} : r = {r:.3f}")

---
## 7. Bilan Jour 1

**Livrables produits :**
- `data/raw/{key}.csv` — 5 séries brutes Eurostat (2010-2025)
- `data/processed/macro_monthly.csv` — DataFrame mensuel consolidé
- `data/processed/narrative_bce_plotly.png` — graphique narratif

**Modules `src/` opérationnels :**
- `config.py` — catalogue datasets + événements BCE
- `fetch_eurostat.py` — `EurostatClient` avec cache CSV
- `transform.py` — `build_macro_df` (fréquences mixtes → mensuel)

**Narrative bancaire validée :**  
BCE 2022-2024 → OLO + EURIBOR montent, inflation culmine, chômage stable.  
Corrélations OLO/EURIBOR/inflation mesurées et visualisées.

**Suite — Jour 2 :**
- `src/persist.py` — stocker le DataFrame dans DuckDB
- Ajouter crédit à la consommation (Eurostat `mir_ir_fir`) et faillites (Statbel)
- Début `app/Home.py` Streamlit

In [ ]:
import os

raw_files = list(DATA_RAW.glob("*.csv"))
proc_files = list(DATA_PROCESSED.glob("*"))

print("data/raw/ :")
for f in raw_files:
    print(f"  {f.name:<35} {f.stat().st_size:>8} bytes")

print("\ndata/processed/ :")
for f in proc_files:
    print(f"  {f.name:<35} {f.stat().st_size:>8} bytes")